# MessMind — Data Science Analysis Notebook
**Predict. Prevent. Preserve.** AI-powered food-waste prediction for hostel messes.

This notebook walks through the full pipeline used by the web app:
1. Collect → 2. Clean → 3. Explore (EDA) → 4. Feature engineering → 5. Model training → 6. Evaluation → 7. Recommendations

> All data is **synthetic** (180 days × 2 meals × 3 messes). Metrics are not evidence of real-world accuracy.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from messmind import data_generator as G
from messmind.preprocessing import clean, add_features
from messmind.models import DemandModel
from messmind import analytics as A
sns.set_theme(style='whitegrid')
pd.set_option('display.width', 160)

## 1. Collect
Generate the raw tables and inject realistic data-quality problems (missing values, grams vs kg, inconsistent spelling, typos, duplicates).

In [ ]:
meals = G.generate_meal_records()
raw = G.inject_dirty_data(meals)
print(raw.shape)
raw.isna().sum()

In [ ]:
raw.head(10)

## 2. Clean

In [ ]:
df, report = clean(raw)
pd.DataFrame(report['steps'])

In [ ]:
print('rows in:', report['rows_in'], '→ rows out:', report['rows_out'])
df.describe().round(2)

## 3. Exploratory Data Analysis

In [ ]:
a = df[df.mess == 'Hostel A']
A.kpis(a)

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
a.groupby('dish')['waste_pct'].mean().sort_values().plot.barh(ax=ax, color='#2f7d5b')
ax.set_title('Most- and least-wasted dishes (avg % of prepared left over)')
plt.show()

In [ ]:
t = a.groupby('date')[['prepared_kg','consumed_kg']].sum().rolling(7).mean()
t.plot(figsize=(11,4), title='Prepared vs consumed (7-day mean)'); plt.show()

In [ ]:
plt.figure(figsize=(8,4.5))
sns.regplot(data=a, x='attendance', y='consumed_kg', scatter_kws={'alpha':.4})
plt.title(f"Attendance vs consumption, r = {a.attendance.corr(a.consumed_kg):.2f}"); plt.show()

In [ ]:
plt.figure(figsize=(8,4.5))
tmp = a.assign(day=a.date.dt.day_name().str[:3])
sns.boxplot(data=tmp, x='day', y='leftover_kg', hue='meal_type', order=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
plt.title('Leftovers by weekday and meal'); plt.show()

In [ ]:
waste = G.generate_waste_breakdown(meals)
waste.groupby('category').waste_kg.sum().plot.pie(autopct='%1.0f%%', figsize=(6,6), title='Waste categories'); plt.ylabel(''); plt.show()

In [ ]:
feedback = G.generate_feedback(meals)
pv = feedback.pivot_table(index='dish', columns='rating', values='student_id', aggfunc='count', fill_value=0)
sns.heatmap(pv.div(pv.sum(1), axis=0)*100, annot=True, fmt='.0f', cmap='YlGn'); plt.title('Preference heatmap (% of responses)'); plt.show()

## 4. Feature engineering

In [ ]:
feats = add_features(df)
fa = feats[feats.mess=='Hostel A']
cols = ['attendance','prepared_kg','consumed_kg','leftover_kg','rating','event_flag','holiday_flag','is_weekend','prev_meal_consumption','rolling7_consumption']
plt.figure(figsize=(9,7)); sns.heatmap(fa[cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0); plt.title('Correlation heatmap'); plt.show()

## 5. Model training
Target = `consumed_kg`. Chronological 80/20 split; Linear Regression, Random Forest and XGBoost (or Gradient Boosting fallback); 5-fold TimeSeriesSplit CV.

In [ ]:
model = DemandModel(df, mess='Hostel A')
s = model.summary()
print('train/test:', s['n_train'], s['n_test'], '| test starts', s['split_date'])
pd.DataFrame(s['results']).drop(columns='train')

In [ ]:
pd.DataFrame(s['baselines'])

## 6. Evaluation

In [ ]:
tf = model.test_frame
plt.figure(figsize=(11,4))
plt.plot(tf.actual.values, label='Actual'); plt.plot(tf.predicted.values, label='Predicted (RF)')
plt.fill_between(range(len(tf)), tf.lower, tf.upper, alpha=.2, label='80% interval')
plt.legend(); plt.title('Actual vs predicted on unseen test data'); plt.show()
print('Interval coverage:', s['interval_coverage'])

In [ ]:
pd.DataFrame(s['importance']).plot.barh(x='feature', y='importance', figsize=(7,5), legend=False, title='Random Forest feature importance'); plt.gca().invert_yaxis(); plt.show()

In [ ]:
plt.figure(figsize=(8,4.5))
sns.histplot(tf.residual, kde=True); plt.axvline(0, color='k'); plt.title('Residuals'); plt.show()

## 7. Forecast & recommendations

In [ ]:
pd.DataFrame(model.forecast(7))[['day','meal_type','dish','attendance','predicted_kg','lower_kg','upper_kg','recommended_prep_kg','usual_prep_kg','potential_saving_kg']]

In [ ]:
inv = A.inventory_status(G.generate_inventory())
recs = A.recommendations(model.forecast(7), inv, A.dish_waste(a), feedback, G.generate_votes())
for r in recs: print(f"[{r['priority']:6}] {r['type']:16} {r['title']}")

In [ ]:
A.sustainability(model.test_frame, a)['food_saved_kg'], A.sustainability(model.test_frame, a)['note']